In [ ]:
with open('../data/input.txt', 'r') as file:
    text = file.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [6]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [7]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target is {target}")

when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [11]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    random_indices = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in random_indices])
    y = torch.stack([data[i+1:i+block_size+1] for i in random_indices])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context} the target is {target}")

inputs:
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets:
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
------
when input is tensor([53]) the target is 59
when input is tensor([53, 59]) the target is 6
when input is tensor([53, 59,  6]) the target is 1
when input is tensor([53, 59,  6,  1]) the target is 58
when input is tensor([53, 59,  6,  1, 58]) the target is 56
when input is tensor([53, 59,  6,  1, 58, 56]) the target is 47
when input is tensor([53, 59,  6,  1, 58, 56, 47]) the target is 40
when input is tensor([53, 59,  6,  1, 58, 56, 47, 40]) the target is 59
when input is tensor([49]) the target is 43
when input is tensor([49, 43]) the target is 43
when input is tensor([49, 43, 43]) the target 

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class Transformer(nn.Module):
    def __init__(self, n_embed=32):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T))  # (T,C)
        x = tok_emb + pos_emb
        logits = self.lm_head(x)  # (B,T,vocab_size)
        
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)

        return idx


m = Transformer()
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, 100)[0].tolist()))

torch.Size([256, 65])
tensor(4.6033, grad_fn=<NllLossBackward0>)

SmVddomOVWydk3'BrlK
QduWGGHPmiu&UXBlIHTZ'yfsDuEtqWPUlOZt&-lV&qBohwN.l;N3z:miimwvg,gAo3EPN3hOw$!VyTuE


In [ ]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(loss.item(), end='\r', flush=True)

In [43]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), 400)[0].tolist()))


Tonilak Bur s thupow ced wolous, otoe ic parl gum haagr t w fune hen seringou INVat? US:
Wo bof,
Whind t mye threntonoshe, whiks d.
D n;
'lsp
Thad! sot!
Yod t ou thipleain du t an.
ONIUCIORENe Leste t, ce, m tanervinchalispres l:
Ofr, ther ionomay:
FLOMou IEETIINO:
HE wee hemans o wlake, be!
Wh y tispoina'd?
Iin dous mefthar
CHiselirel;
K: se hon, I e way I t it as wh s,
QUn noutheat
TI C'de t TIn


In [49]:
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)
q = query(x)
v = value(x)

w = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, 16) @ (B, 16, T) -> (B, T, T)

tril = torch.tril(torch.ones(T, T))
w = w.masked_fill(tril == 0, float('-inf'))
w = F.softmax(w, dim=-1)

out = w @ v

print(out.shape)
print(w[0])

torch.Size([4, 8, 16])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5806, 0.4194, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2298, 0.2961, 0.4741, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1850, 0.1575, 0.1994, 0.4582, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2363, 0.1441, 0.1527, 0.2041, 0.2628, 0.0000, 0.0000, 0.0000],
        [0.0682, 0.1363, 0.2922, 0.1303, 0.2138, 0.1592, 0.0000, 0.0000],
        [0.1256, 0.1644, 0.0922, 0.1361, 0.2630, 0.1015, 0.1173, 0.0000],
        [0.0887, 0.0874, 0.1539, 0.0789, 0.0664, 0.1493, 0.1848, 0.1906]],
       grad_fn=<SelectBackward0>)


In [105]:
block_size = 256
batch_size = 64
dropout = 0.2
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
device

device(type='mps')

In [106]:
class Head(nn.Module):
    def __init__(self, head_size, n_embed):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size, device=device)))

    def forward(self, x):
        _,T,_ = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        w = q @ k.transpose(-2, -1) * self.head_size**-0.5
        w = w.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        w = F.softmax(w, dim=-1)
        w = self.dropout(w)
        out = w @ v

        return out

In [107]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, head_size, n_embed):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embed) for _ in range(n_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [108]:
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [109]:
class Block(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size, n_embed)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
        
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [110]:
class Transformer(nn.Module):
    def __init__(self, n_embed=384, n_heads=6, n_blocks=6):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList([Block(n_embed, n_heads) for _ in range(n_blocks)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)  # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T).to(device))  # (T,C)
        x = tok_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B,T,vocab_size)
        
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=-1)

        return idx


m = Transformer().to(device)
xb = xb.to(device)
yb = yb.to(device)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

idx = torch.zeros((1, 1), dtype=torch.long).to(device)
print(decode(m.generate(idx, 100)[0].tolist()))

torch.Size([128, 65])
tensor(4.2775, device='mps:0', grad_fn=<NllLossBackward0>)

llGQZFfxr$rvYnbx A3:WDGxOdjrsevL3dDHLu
jhhpYSJaFBW.xSHjubcX&b?oTkw!smwE:fP-PM3UGE,j-?gq-NNZeSXIwpjks


In [125]:
from tqdm import tqdm

optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)
batch_size = 64
n_iter = 100
for steps in tqdm(range(n_iter)):
    xb, yb = get_batch('train')
    xb = xb.to(device)
    yb = yb.to(device)
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item(), end='\r', flush=True)
torch.save(m.state_dict(), 'nanogpt.pth')

100%|██████████| 100/100 [02:27<00:00,  1.48s/it]

In [129]:
# Print in a streaming fashion
n_tokens = 400
with torch.no_grad():
    m.eval()
    input = torch.zeros((1, 1), dtype=torch.long).to(device)
    for _ in range(n_tokens):
        logits, _ = m(input[:, -block_size:])
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        input = torch.cat((input, idx_next), dim=-1)
        print(decode([idx_next.item()]), end='', flush=True)
    m.train()

His his mothind, this
Romeo bast; he come Eomend,
And end in yours, you have long and gracio
And is thou wind be this both.

ESCALUS:
They, I say hear sin tell be remptent:
I twill your right him to wouldre streath'd
And be of you bear nor like him of prince.

Nurse:
My Vale---
You; ne's resanion, delive flows.

POMPEY:
UnsERCUSAN:
Their of when?

LEONTES:
Ha!
I friend madard thy climps!
Make you 